
# Tools

**Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:**

1.A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2.A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3.6-27b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Question**: The user asks "Why do parrots talk?" This is a common question about animal behavior, specifically avian vocalization and mimicry.\n\n2.  **Identify Key Concepts**:\n   - Parrots are known for mimicking human speech\n   - It\'s not "talking" in the human sense (language comprehension/communication)\n   - It\'s vocal mimicry\n   - Evolutionary/behavioral reasons: social bonding, territory, mate attraction, flock integration\n   - Biological mechanisms: syrinx, brain structure (vocal learning centers)\n   - Environmental factors: captivity vs. wild, human interaction\n\n3.  **Structure the Answer**:\n   - Clarify the misconception (they don\'t "talk" like humans)\n   - Explain the biological capability (vocal learning, syrinx, brain)\n   - Explain the evolutionary/behavioral reasons (social bonding, flock dynamics, survival)\n   - Explain why they mimic humans specifically (captivity, social re

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """ Get the weather at a location"""
    return f"It's Sunny in {location}"

model_with_tools=model.bind_tools([get_weather])

In [3]:
response=model_with_tools.invoke("What is the weather in Howrah?")
print(response)


content='' additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  Identify the user\'s request: Weather in Howrah.\n2.  Identify the available tool: `get_weather` which takes a `location` parameter.\n3.  Extract the location: "Howrah".\n4.  Call the tool: `get_weather(location="Howrah")`.\n5.  Formulate the response based on the tool\'s output. (Wait for tool output) -> I will generate the tool call now.✅\n', 'tool_calls': [{'id': 'pjyhepr35', 'function': {'arguments': '{"location":"Howrah"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 129, 'prompt_tokens': 276, 'total_tokens': 405, 'completion_time': 0.245942102, 'completion_tokens_details': {'reasoning_tokens': 100}, 'prompt_time': 0.02187603, 'prompt_tokens_details': None, 'queue_time': 0.05140316, 'total_time': 0.267818132}, 'model_name': 'qwen/qwen3.6-27b', 'system_fingerprint': 'fp_fff3b79855', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs

### Tool Execution Loops

In [4]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Howrah?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

It's currently sunny in Howrah.


In [5]:
messages

[{'role': 'user', 'content': "What's the weather in Howrah?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  Identify the user\'s intent: Get weather information.\n2.  Identify the location: Howrah.\n3.  Check available tools: `get_weather` takes a `location` parameter.\n4.  Call the tool: `get_weather(location="Howrah")`.\n5.  Formulate response based on tool output. (Wait for tool output) -> I will simulate the tool call.\n\nLet\'s call the tool. \n`get_weather(location="Howrah")`\nI will generate the tool call. \nNote: Howrah is a city in West Bengal, India. The tool should handle it.\nProceed. \nTool call format: `call: default_api:get_weather{"location": "Howrah"}` (or similar depending on the system, but I\'ll just output the standard format).\nActually, I\'ll just output the function call as requested.\nWait, the prompt says "If you choose to call a function ONLY reply in the following format with NO suffix: <tool_call>\n<function=example

#### Explore Langchain tool Documentation
https://docs.langchain.com/oss/python/langchain/tools#tools

### mini project: weather API Integration

In [6]:
import os
import requests
from dotenv import load_dotenv
from langchain.tools import tool

load_dotenv()

API_KEY = os.getenv("OPENWEATHER_API_KEY")


@tool
def get_weather(city: str) -> str:
    """Get the current weather information for a city."""

    url = "https://api.openweathermap.org/data/2.5/weather"

    params = {
        "q": city,
        "appid": API_KEY,
        "units": "metric"
    }

    response = requests.get(url, params=params)

    if response.status_code != 200:
        return f"Could not get weather information for {city}."

    data = response.json()

    temperature = data["main"]["temp"]
    description = data["weather"][0]["description"]

    return f"The weather in {city} is {temperature}°C with {description}."

In [7]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3.6-27b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Question**: The user asks "Why do parrots talk?" This is a straightforward biological/behavioral question about parrot vocalization, specifically mimicry and speech-like behavior.\n\n2.  **Identify Key Concepts**:\n   - Parrots don\'t actually "talk" in the human sense (they don\'t understand language semantically like humans do)\n   - They are highly skilled vocal mimics\n   - Evolutionary/biological reasons for mimicry\n   - Social/behavioral functions in the wild\n   - Captivity vs. wild contexts\n   - Neurological/anatomical adaptations (syrinx, brain regions like nidopallium caudolaterale)\n   - Reinforcement learning (they learn what gets attention/rewards)\n\n3.  **Structure the Response**:\n   - Clarify what "talking" really means for parrots\n   - Explain the biological/anatomical basis\n   - Discuss evolutionary/social reasons (wild context)\n   - Explain how captivity changes the behavior\n   

In [8]:
model_with_tools=model.bind_tools([get_weather])

In [10]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in kolkata?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The weather in Kolkata is currently 29.97°C with light rain.
